# Week 7: Deep Learning Foundations
## Notebook 10: Gated Recurrent Unit (GRU) Forward & Backprop from Scratch

### 1. Conceptual Intuition & Theoretical Background
Standard Recurrent Neural Networks (RNNs) suffer from vanishing gradients across long sequences because the recurrent Jacobian matrix $\frac{\partial h_t}{\partial h_{t-1}}$ repeatedly multiplies weight matrices.

The **Gated Recurrent Unit (GRU)** solves this with an additive shortcut pathway while using fewer parameters than an LSTM (2 gates instead of 3, eliminating the separate cell state $c_t$):
1. **Reset Gate ($r_t$):** Controls how much past hidden state to forget:
$$r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$$
2. **Update Gate ($z_t$):** Balances past information with new candidate activation:
$$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)$$
3. **Candidate State ($\tilde{h}_t$):**
$$\tilde{h}_t = \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h)$$
4. **Final Hidden State ($h_t$):**
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

---

### 2. Scratch Python Implementation: GRU Cell

In [ ]:
import numpy as np

class GRUCellScratch:
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        # Xavier initialization
        limit = np.sqrt(6.0 / (input_dim + hidden_dim))
        self.W_z = np.random.uniform(-limit, limit, (hidden_dim, input_dim))
        self.U_z = np.random.uniform(-limit, limit, (hidden_dim, hidden_dim))
        self.b_z = np.zeros((hidden_dim, 1))
        
        self.W_r = np.random.uniform(-limit, limit, (hidden_dim, input_dim))
        self.U_r = np.random.uniform(-limit, limit, (hidden_dim, hidden_dim))
        self.b_r = np.zeros((hidden_dim, 1))
        
        self.W_h = np.random.uniform(-limit, limit, (hidden_dim, input_dim))
        self.U_h = np.random.uniform(-limit, limit, (hidden_dim, hidden_dim))
        self.b_h = np.zeros((hidden_dim, 1))
        
    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -25, 25)))
        
    def forward(self, x_t, h_prev):
        """
        x_t: (input_dim, batch_size)
        h_prev: (hidden_dim, batch_size)
        """
        # Update Gate
        z_t = self._sigmoid(self.W_z @ x_t + self.U_z @ h_prev + self.b_z)
        
        # Reset Gate
        r_t = self._sigmoid(self.W_r @ x_t + self.U_r @ h_prev + self.b_r)
        
        # Candidate Hidden State
        h_tilde = np.tanh(self.W_h @ x_t + self.U_h @ (r_t * h_prev) + self.b_h)
        
        # Hidden State interpolation
        h_t = (1.0 - z_t) * h_prev + z_t * h_tilde
        return h_t, (z_t, r_t, h_tilde)

# Test forward pass over sequence of length T=5
input_dim = 10
hidden_dim = 16
batch_size = 4
seq_len = 5

gru = GRUCellScratch(input_dim, hidden_dim)
h = np.zeros((hidden_dim, batch_size))

# Simulate sequential processing
for t in range(seq_len):
    x_t = np.random.randn(input_dim, batch_size)
    h, cache = gru.forward(x_t, h)

print(f"Final Hidden State Shape: {h.shape}")
print(f"Mean Hidden State Activation: {np.mean(h):.4f}")
print("GRU forward propagation through time completed smoothly!")

### 3. Step-by-Step Code Walkthrough
1. **Linear Gate Transformations**: Evaluates reset and update gates using parallel matrix multiplications.
2. **Gated Candidate Vector**: Incorporates past history scaled by $r_t$, letting the network ignore irrelevant prior steps.
3. **Linear Convex Combination**: When $z_t = 0$, $h_t = h_{t-1}$, creating a direct identity highway for gradient backpropagation.

---

### 4. Industry Applications & Trade-offs
- **Edge Device Speech Recognition:** GRU models require $25\%$ fewer weights than LSTMs, speeding up on-device audio transcription.
- **Trade-off:** LSTMs remain superior on long-range associative recall tasks due to dedicated linear memory cells.

---

### 5. Top Interview Questions & Answers
**Q1: What are the structural differences between GRU and LSTM?**
> **Answer:** GRU combines the input and forget gates into a single update gate ($z$), merges cell state and hidden state into a single recurrent state ($h$), and has fewer trainable parameters than an LSTM.

**Q2: How does the GRU update gate resolve vanishing gradients?**
> **Answer:** When $z_t \approx 0$, the hidden state updates as $h_t \approx h_{t-1}$. In backpropagation, $\frac{\partial h_t}{\partial h_{t-1}} \approx 1$, allowing gradients to flow backward through arbitrary time steps without exponential decay.